# L2-ARCTIC data prep for POWSM LoRA (perceived-only)

Builds 20 s / 16 kHz chunks from the **manually annotated** `annotation/` TextGrids
(~150 utterances per speaker), resolving the `CPL,PPL,s/d/a` error templates to the
**perceived** (L2-surface) phone sequence, then mapping ARPABET -> POWSM IPA.

**Before running:** download L2-ARCTIC and point `L2ARCTIC_RAW_DIR` at its root (the folder
containing `ABA/`, `SKA/`, ...). Raw audio is never committed (`data/` is gitignored).

Outputs to `sig/fine-tune/data/l2arctic_chunks/`: `<id>.wav`, `manifest.json`,
`train/val/test.json`, `prep_audit.json`, `oov_report.csv`.

**Local:** set cwd to `sig/fine-tune` or `sig/fine-tune/notebooks` so the import resolves.


In [6]:
from __future__ import annotations
import json, os, sys
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "l2arctic_util.py").is_file():
    FT = _here
elif (_here.parent / "l2arctic_util.py").is_file():
    FT = _here.parent
else:
    raise SystemExit("Run with cwd = sig/fine-tune or sig/fine-tune/notebooks")
sys.path.insert(0, str(FT))

import l2arctic_util as L

# Point this at your downloaded L2-ARCTIC root (folder of ABA/, SKA/, ...).
RAW_DIR = Path(os.environ.get("L2ARCTIC_RAW_DIR", FT / "l2arctic_raw"))
OUT_DIR = L.DEFAULT_CHUNKS_DIR
print("RAW_DIR:", RAW_DIR, "| exists:", RAW_DIR.is_dir())
print("OUT_DIR:", OUT_DIR)
if RAW_DIR.is_dir():
    found = [d.name for d in sorted(RAW_DIR.iterdir()) if d.is_dir() and d.name.upper() in L.L2ARCTIC_SPEAKER_L1]
    print("known speaker dirs:", found)


RAW_DIR: C:\Users\faruq\Desktop\college\senior\sig\fine-tune\l2arctic_raw | exists: True
OUT_DIR: C:\Users\faruq\Desktop\college\senior\sig\fine-tune\data\l2arctic_chunks
known speaker dirs: ['ABA', 'ASI', 'BWC', 'EBVS', 'ERMS', 'HJK', 'HKK', 'HQTV', 'LXC', 'MBMPS', 'NCC', 'NJS', 'PNV', 'RRBI', 'SKA', 'SVBI', 'THV', 'TLV', 'TNI', 'TXHC', 'YBAA', 'YDCK', 'YKWK', 'ZHAA']


In [7]:
mapping, silence = L.load_arpabet_map()
audit = L.new_audit()
manifest = L.build_l2arctic_manifest(RAW_DIR, OUT_DIR, mapping, silence, audit=audit)

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
n_tr, n_va, n_te = L.write_splits(OUT_DIR, manifest)
fa = L.finalize_audit(audit)
(OUT_DIR / "prep_audit.json").write_text(json.dumps(fa, ensure_ascii=False, indent=2), encoding="utf-8")
L.write_oov_report(OUT_DIR / "oov_report.csv", audit)

print(f"utterances written : {len(manifest)}")
print(f"train/val/test     : {n_tr}/{n_va}/{n_te}")
print(f"dropped (empty)    : {fa['chunks_dropped_empty']}")
print(f"truncated (>20 s)  : {fa['utts_truncated']}")
print(f"error-type counts  : {fa['error_type_counts']}")
print(f"OOV rate           : {fa['oov_rate']*100:.2f}%  (target < 2%)")
print(f"top OOV tokens     : {list(fa['oov_token_counts'].items())[:15]}")
print(f"unique IPA phones  : {len(fa['final_ipa_counts'])}")


utterances written : 3599
train/val/test     : 1800/899/900
dropped (empty)    : 0
truncated (>20 s)  : 0
error-type counts  : {'plain': 101633, 'substitution': 14098, 'deletion': 3420, 'addition': 1092}
OOV rate           : 1.52%  (target < 2%)
top OOV tokens     : [('R*', 489), ('ERR', 269), ('AA*', 181), ('T*', 174), ('D*', 88), ('HH*', 86), ('L*', 68), ('EH*', 57), ('N*', 54), ('ER*', 54), ('OW*', 48), ('P*', 34), ('B*', 31), ('W*', 26), ('V*', 25)]
unique IPA phones  : 35


In [8]:
import soundfile as sf
bad = []
for c in manifest[:50]:
    a, r = sf.read(OUT_DIR / f"{c['id']}.wav")
    if r != 16000 or a.shape[0] != 320_000:
        bad.append((c["id"], r, a.shape[0]))
print("WAV sanity (first 50):", "ok" if not bad else bad)


WAV sanity (first 50): ok


In [9]:
# Requires espnet + espnet_model_zoo. Confirms every target IPA token is in POWSM's vocab
# (slash-tokens). Anything printed here must be added to arpabet_to_ipa.json before training.
from espnet2.bin.s2t_inference import Speech2Text

s2t = Speech2Text.from_pretrained("espnet/powsm", device="cpu", lang_sym="<unk>", task_sym="<pr>")
vocab = set(s2t.converter.token2id.keys())
powsm_phones = {t.strip("/") for t in vocab if t.startswith("/") and t.endswith("/")}
ipa_phones = {p for c in manifest for p in c["phones"]}
unknown = sorted(ipa_phones - powsm_phones)
print("unique IPA phones in manifest:", len(ipa_phones))
print("NOT in POWSM vocab:", unknown if unknown else "none")


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

unique IPA phones in manifest: 35
NOT in POWSM vocab: none
